# Train an SO101-Nexus policy with demo-seeded PPO on MuJoCo Warp

Bootstrap a policy from 10 teleop demos, then fine-tune it with reinforcement learning. Run every cell top to bottom.

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so this clones the repo and installs the `warp` + `train` extras.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip uninstall -y so101-nexus 2>/dev/null || true
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"
# `pip install -e` registers the package via a .pth file that Python only
# reads when an interpreter starts. This kernel was already running before
# the install above, so `!python ...` cells (fresh subprocesses) see the
# package but in-kernel `import so101_nexus...` cells below will not unless
# `src/` is also put on sys.path explicitly, here.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 3. Choose the task and seed

`WarpPickLift-v1` (`"pick_lift"`) is the original, fastest-to-train task (~30 min). `WarpPickAndPlace-v1` (`"pick_and_place"`) is harder (carry the object onto a goal disc, lower it, and hold still) and takes much longer (~5.5h on an RTX 5090). For pick-lift, seed 5 reproduces our best result. Pick-and-place is validated across seeds 1-3 (`best_success` mean `0.86`, std `0.08`); any seed works reasonably well but seed choice matters more for this harder task than for pick-lift.

In [ ]:
TASK = "pick_lift"  # "pick_lift" or "pick_and_place"
SEED = 5

if TASK == "pick_lift":
    ENV_ID = "WarpPickLift-v1"
    DEMO_REPO = "johnsutor/MuJoCoPickLift-v1"
    SUCCESS_BONUS = 0.0
    STEPS = 30_000_000
    ANNEAL_TIMESTEPS = 0
    LR_MIN_FRAC = 0.0
elif TASK == "pick_and_place":
    ENV_ID = "WarpPickAndPlace-v1"
    DEMO_REPO = "johnsutor/MuJoCoPickAndPlace-v1"
    SUCCESS_BONUS = 50.0
    STEPS = 160_000_000
    ANNEAL_TIMESTEPS = 80_000_000
    LR_MIN_FRAC = 0.1
else:
    raise ValueError(f"unknown TASK {TASK!r}; use 'pick_lift' or 'pick_and_place'")

print(f"Training {ENV_ID} for {STEPS:,} env steps, seed={SEED}")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

Runs demo-seeded PPO for `STEPS` env steps (30M for pick-lift, ~30 min; 160M for pick-and-place, ~5.5h). Takes a while, so grab a coffee.

In [ ]:
!python examples/bc_ppo_warp.py --exp-name colab --seed {SEED} --env-id {ENV_ID} \
    --demo-repo {DEMO_REPO} --success-bonus {SUCCESS_BONUS} --total-timesteps {STEPS} \
    --anneal-timesteps {ANNEAL_TIMESTEPS} --lr-min-frac {LR_MIN_FRAC} --eval-freq 0

## 6. Evaluate the trained policy

Runs a deterministic evaluation across 512 parallel Warp environments.

In [ ]:
CKPT = f"runs/{ENV_ID}__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id {ENV_ID} --checkpoint "{CKPT}"

## 7. Watch a sample rollout

Renders one episode of the trained policy so you can see it work.


In [ ]:
# Render one deterministic rollout of the trained policy and play it inline.
import glob

from IPython.display import Video

from examples.bc_ppo_warp import rollout_video_from_checkpoint

checkpoint = sorted(glob.glob(CKPT))[-1]
print(f"Rendering sample rollout from {checkpoint}")

metrics, video_path = rollout_video_from_checkpoint(
    checkpoint,
    ENV_ID,
    control_mode="pd_joint_delta_pos",
    episode_length=512,
    hidden_dim=256,
    seed=12345,
)
print(
    f"rollout success_rate={metrics['eval/success_rate']:.2f} "
    f"return={metrics['eval/return']:.2f} "
    f"ep_len={metrics['eval/ep_len']:.0f}"
)
Video(video_path)

## Next steps

- Compare against vanilla PPO by rerunning with `--use-demos false`.
- Full walkthrough: [Training with PPO](https://so101-nexus.com/docs/training/ppo).